In [1]:
import requests
import zipfile
import pandas as pd

# extração
url = "https://infosiga.detran.sp.gov.br/rest/painel/download/file/dados_infosiga.zip"
caminho_zip = "/lakehouse/default/Files/silver_seguranca_viaria/dados_infosiga_janeiro26.zip"
pasta_extracao = "/lakehouse/default/Files/silver_seguranca_viaria/dados_infosiga/"

resp = requests.get(url)
resp.raise_for_status()

with open(caminho_zip, "wb") as f:
    f.write(resp.content)

with zipfile.ZipFile(caminho_zip, 'r') as zip_ref:
    zip_ref.extractall(pasta_extracao)

StatementMeta(, 09afcde7-50a9-4def-a7ea-3ffa9641fa64, 3, Finished, Available, Finished, False)

In [2]:
# leitura + consolidação
pessoas21 = pd.read_csv(
    pasta_extracao + "/pessoas_2015-2021.csv", encoding="latin1", sep=";"
)
pessoas26 = pd.read_csv(
    pasta_extracao + "/pessoas_2022-2026.csv", encoding="latin1", sep=";"
)
pessoas = pd.concat([pessoas21, pessoas26], axis=0)

sinistros21 = pd.read_csv(
    pasta_extracao + "/sinistros_2015-2021.csv", encoding="latin1", sep=";"
)
sinistros26 = pd.read_csv(
    pasta_extracao + "/sinistros_2022-2026.csv", encoding="latin1", sep=";"
)
sinistros = pd.concat([sinistros21, sinistros26], axis=0)

veiculos21 = pd.read_csv(
    pasta_extracao + "/veiculos_2015-2021.csv", encoding="latin1", sep=";"
)
veiculos26 = pd.read_csv(
    pasta_extracao + "/veiculos_2022-2026.csv", encoding="latin1", sep=";"
)
veiculos = pd.concat([veiculos21, veiculos26], axis=0)

pessoas.to_parquet(
    pasta_extracao + "silver_infosiga_pessoas.parquet", index=False
)
sinistros.to_parquet(
    pasta_extracao + "silver_infosiga_sinistros.parquet", index=False
)
veiculos.to_parquet(
    pasta_extracao + "silver_infosiga_veiculos.parquet", index=False
)

StatementMeta(, 09afcde7-50a9-4def-a7ea-3ffa9641fa64, 4, Finished, Available, Finished, False)

In [3]:
pessoas_sdf = spark.createDataFrame(pessoas)
(
    pessoas_sdf
    .write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema","true")
    .saveAsTable("silver_infosiga_pessoas")
)

veiculos_sdf = spark.createDataFrame(veiculos)
(
    veiculos_sdf
    .write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema","true")
    .saveAsTable("silver_infosiga_veiculos")
)

sinistros_sdf = spark.createDataFrame(sinistros)
(
    sinistros_sdf
    .write.mode("overwrite")
    .format("delta")
    .option("overwriteSchema","true")
    .saveAsTable("silver_infosiga_sinistros")
)

StatementMeta(, 09afcde7-50a9-4def-a7ea-3ffa9641fa64, 5, Finished, Available, Finished, True)